In [0]:
%run ./01_setup_environment

In [0]:
# ========================================
# 14_icu_alert_pipeline
# ========================================

from pyspark.sql.functions import *

stream_df = (
    spark.readStream
        .format("delta")
        .load(f"{bronze_path}/icu_stream")
)

alert_df = (
    stream_df.filter(
        (col("heart_rate") > 120) |
        (col("oxygen_level") < 90)
    )
)

query = (
    alert_df.writeStream
        .format("delta")
        .outputMode("append")
        .trigger(availableNow=True)  # Free Edition compatible
        .option(
            "checkpointLocation",
            f"{checkpoint_path}/icu_alerts"
        )
        .start(f"{gold_path}/icu_alerts")
)

query.awaitTermination()